# Problema del Peluquero Dormido — Análisis Formal y Verificación Experimental

**Universidad del Valle — Departamento de Ingeniería de Sistemas**  
**Sistemas Operativos — Taller Final**  

---

## Frente 1 — Reproducción Determinista del Fallo (*Lost Wakeup*)

### 1. Objetivo
El objetivo de esta sección es reproducir e ilustrar de manera completamente determinista el fallo de concurrencia conocido como **Lost Wakeup** (Despertar Perdido) en la versión incorrecta del problema del peluquero dormido, donde no se emplea una sincronización atómica para la secuencia de verificación de estado y bloqueo (dormir).

## 2. Fundamento Teórico

El fenómeno del **Lost Wakeup** ocurre cuando un hilo lector/receptor se prepara para bloquearse esperando una señal, pero justo antes de efectuar la llamada de bloqueo (p. ej., `wait()`), es interrumpido por el planificador (scheduler). Otro hilo (el emisor) se ejecuta, altera la condición de espera y envía la señal de despertar (`signal` o `set`). Como el primer hilo aún no está técnicamente en estado de espera, la señal se pierde. Cuando el primer hilo reanuda su ejecución, procede a bloquearse y queda suspendido indefinidamente porque la señal ya fue emitida y no volverá a ocurrir.

En el problema del peluquero dormido:
1. El barbero comprueba `waiting == 0` (no hay clientes).
2. Justo antes de irse a dormir (ejecutar `wait()`), llega un cliente.
3. El cliente ve que el barbero está "despierto" (pues el estado aún no es "durmiendo"), por lo que no le envía la señal de despertar (`set()`), limitándose a sentarse en la silla de espera.
4. El barbero reanuda su ejecución, asume que no hay clientes porque ya hizo la evaluación previa, y se duerme (`wait()`).
5. El barbero queda durmiendo indefinidamente (Lost Wakeup) y el cliente esperando en la silla para siempre.

In [ ]:
# ===========================================================================
# FRENTE 1 — Implementación INCORRECTA del Problema del Peluquero Dormido
# Propósito: Demostrar el Lost Wakeup de forma determinista
# ===========================================================================

import threading
import time
import random

# Parámetros de la barbería
NUM_CHAIRS = 3          # Número de sillas de espera
INJECT_DELAY = 0.15     # Retardo inyectado para simular la interrupción del scheduler

# Variables de estado compartidas sin protección adecuada de exclusión mutua
waiting = 0             # Clientes esperando
barber_sleeping = False # Estado del barbero
barber_event = threading.Event()  # Evento para despertar al barbero

# Registro de eventos
start_time = time.perf_counter()
event_log = []
log_lock = threading.Lock()  # Exclusión mutua únicamente para el orden del log impreso

def log(actor, message):
    """Registra un evento con marca de tiempo precisa."""
    with log_lock:
        ts = time.perf_counter() - start_time
        entry = f"[T={ts:.4f}s] [{actor}] {message}"
        event_log.append(entry)
        print(entry)

def barber_incorrect():
    global waiting, barber_sleeping
    
    log("BARBERO", "Inicio de turno. Verificando si hay clientes...")
    
    # Verificación del estado de la sala
    # Entre esta lectura de 'waiting' y el bloqueo físico en 'wait()'
  
    if waiting == 0:
        log("BARBERO", "No hay clientes. Me preparo para dormir...")
        
        # INYECCIÓN DEL RETARDO:
        # Forzamos al scheduler a dar paso al cliente antes de bloquearnos.
        time.sleep(INJECT_DELAY)
        
        # Transición al estado durmiendo y bloqueo
        barber_sleeping = True
        log("BARBERO", "Entrando en estado durmiendo. Ejecutando wait()...")
        
        # Usamos un timeout de 2.0 segundos para el experimento
        woke_up = barber_event.wait(timeout=2.0)
        
        if woke_up:
            barber_sleeping = False
            log("BARBERO", "Desperté. Atendiendo al cliente.")
        else:
            log("BARBERO", " FALLO: Nadie me despertó. Lost Wakeup confirmado.")
    else:
        log("BARBERO", f"Hay {waiting} cliente(s) esperando. Atendiendo...")

def client_incorrect(client_id, arrival_delay):
    global waiting, barber_sleeping
    
    # Simula el viaje del cliente a la barbería
    time.sleep(arrival_delay)
    log(f"CLIENTE-{client_id}", "Llegando a la barbería...")
    
    # Condicion para evaluar sillas libres
    if waiting < NUM_CHAIRS:
        waiting += 1
        log(f"CLIENTE-{client_id}", f"Me senté en una silla. Esperando = {waiting}")
        
        # El cliente comprueba si el barbero duerme
        # Como el barbero está en su ventana de retardo, barber_sleeping es aún False.
        # El cliente asume erróneamente que el barbero está despierto y NO envía la señal.
        if barber_sleeping:
            log(f"CLIENTE-{client_id}", "El barbero está durmiendo. Enviando señal de despertar (set)... ")
            barber_event.set()
        else:
            log(f"CLIENTE-{client_id}", "Barbero despierto. NO envío señal. [AQUI OCURRE EL FALLO]")
    else:
        log(f"CLIENTE-{client_id}", "Sala llena. Me retiro.")

### 3. Ejecución Experimental del Fallo

In [145]:
print("=== EXPERIMENTO: REPRODUCCIÓN DEL LOST WAKEUP ===\n")

# Limpieza e inicialización
event_log.clear()
barber_event.clear()
waiting = 0
barber_sleeping = False

# Hilos: El barbero inicia de inmediato (T=0). El cliente llega en T=0.05s, 
# cayendo exactamente en la ventana de vulnerabilidad del barbero (0s a 0.15s).
t_barber = threading.Thread(target=barber_incorrect)
t_client = threading.Thread(target=client_incorrect, args=(1, 0.05))

t_start = time.perf_counter()
t_barber.start()
t_client.start()

t_barber.join()
t_client.join()
t_duration = time.perf_counter() - t_start

print(f"\nDuración de la simulación: {t_duration:.4f} segundos")
if t_duration >= 2.0:
    print("\nRESULTADO: [Fallo Confirmado] El barbero quedó bloqueado permanentemente (se superó el timeout).")
else:
    print("\nRESULTADO: El barbero se despertó a tiempo.")

=== EXPERIMENTO: REPRODUCCIÓN DEL LOST WAKEUP ===

[T=0.0065s] [BARBERO] Inicio de turno. Verificando si hay clientes...
[T=0.0065s] [BARBERO] No hay clientes. Me preparo para dormir...
[T=0.0609s] [CLIENTE-1] Llegando a la barbería...
[T=0.0610s] [CLIENTE-1] Me senté en una silla. Esperando = 1
[T=0.0610s] [CLIENTE-1] Barbero despierto. NO envío señal. [AQUI OCURRE EL FALLO]
[T=0.1615s] [BARBERO] Entrando en estado durmiendo. Ejecutando wait()...
[T=2.1652s] [BARBERO]  FALLO: Nadie me despertó. Lost Wakeup confirmado.

Duración de la simulación: 2.1594 segundos

RESULTADO: [Fallo Confirmado] El barbero quedó bloqueado permanentemente (se superó el timeout).


---

## Frente 2 — Modelo e Invariantes del Sistema

### 1. Modelo Formal del Sistema

Para analizar formalmente la concurrencia en la barbería, definimos los componentes del sistema:

#### 1.1 Actores y sus Estados
- **Barbero:** DURMIENDO, VERIFICANDO, TRABAJANDO.
- **Clientes:** VIAJANDO, LLEGANDO, ESPERANDO, RECIBIENDO_CORTE, ATENDIDO, RECHAZADO.

#### 1.2 Recursos Compartidos y Variables
- **Sala de Espera:** Capacidad limitada de $N$ sillas (recurso crítico).
- `sillas_espera`: Contador de clientes esperando en la sala.
- `barbero_durmiendo`: Bandera lógica del estado del barbero.

--- 

### 2. Invariantes del Sistema (Propiedades de Seguridad)

- **Invariante 1 (Capacidad de la Sala):**
  $$0 \le sillas\_espera \le CAPACIDAD\_MAX$$
- **Invariante 2 (Ausencia de Negligencia del Barbero):**
  $$\text{Si } barbero\_durmiendo = True \implies sillas\_espera = 0$$
- **Invariante 3 (Ausencia de Espera Inútil):**
  $$\text{Si } sillas\_espera > 0 \implies barbero\_durmiendo = False$$

### 3. Implementación Simplificada de Verificación de Invariantes

In [146]:
total_verificaciones = 0

In [188]:
# ===========================================================================
# FRENTE 2 — FUNCIÓN DE VERIFICACIÓN DE INVARIANTES
# ===========================================================================

def verificar_invariantes(sillas_espera, barbero_durmiendo, capacidad_max=3):

    global total_verificaciones
    total_verificaciones += 1

    """Valida las aserciones de seguridad del sistema."""
    # Invariante 1: Capacidad de la sala
    assert 0 <= sillas_espera <= capacidad_max, \
        f"Violación: sillas_espera ({sillas_espera}) fuera de rango [0, {capacidad_max}]."
    
    # Invariante 2: Si el barbero duerme, la sala de espera debe estar vacía
    if barbero_durmiendo:
        assert sillas_espera == 0, \
            f"Violación: El barbero duerme pero hay {sillas_espera} clientes esperando."
            
    # Invariante 3: Si hay clientes esperando, el barbero no puede estar dormido
    if sillas_espera > 0:
        assert not barbero_durmiendo, \
            "Violación: Hay clientes esperando pero el barbero está dormido."

# ===========================================================================
# PRUEBA DE FUNCIONAMIENTO DE LOS INVARIANTES
# ===========================================================================
print("=== PRUEBA DE INVARIANTES (FRENTE 2) ===\n")

# Caso 1: Estado válido
print("1. Evaluando estado válido (1 cliente esperando, barbero despierto)... ")
verificar_invariantes(sillas_espera=1, barbero_durmiendo=False)
print("   Resultado: [OK] Estado seguro.")

# Caso 2: Estado inválido (fuerza un error)
print("\n2. Evaluando estado inválido (2 clientes esperando, barbero dormido)... ")
try:
    verificar_invariantes(sillas_espera=2, barbero_durmiendo=True)
    print("   Resultado: [ERROR] Se permitió un estado inconsistente.")
except AssertionError as e:
    print(f"   Resultado: [VIOLACIÓN DETECTADA CORRECTAMENTE] {e}")

=== PRUEBA DE INVARIANTES (FRENTE 2) ===

1. Evaluando estado válido (1 cliente esperando, barbero despierto)... 
   Resultado: [OK] Estado seguro.

2. Evaluando estado inválido (2 clientes esperando, barbero dormido)... 
   Resultado: [VIOLACIÓN DETECTADA CORRECTAMENTE] Violación: El barbero duerme pero hay 2 clientes esperando.


---

## Frente 3 — Implementación correcta

### 1. Justificación del Mecanismo de Sincronización

Utilizaremos una única variable de condición `threading.Condition()`. Esta primitiva envuelve un lock interno que garantiza exclusión mutua para leer y escribir sobre las variables globales, además de permitir la sincronización mediante `wait()` y `notify_all()` para evitar el Lost Wakeup.

##  Recursos Compartidos y Mecanismos de Sincronización

En este bloque se inicializan los recursos compartidos que utilizarán todos los hilos durante la simulación. Se define el **Lock**, encargado de garantizar la exclusión mutua sobre el estado compartido, y la **Variable de Condición (`Condition`)**, utilizada para coordinar el sueño y el despertar del barbero sin recurrir a espera activa (*Busy Waiting*).

También se inicializan las variables que representan el estado de la barbería, como el número de clientes esperando, el estado del barbero, la apertura de la barbería y los contadores de auditoría que permitirán verificar posteriormente el comportamiento de la simulación.

In [ ]:
# ===========================================================================
# FRENTE 3 — IMPLEMENTACIÓN CORRECTA DEL PROBLEMA DEL PELUQUERO DORMIDO
# Chunk 1: Recursos Compartidos y Mecanismos de Sincronización
# ===========================================================================

import threading
import time
import random

# -------------------------------------------------------------
# Parámetros de la simulación
# -------------------------------------------------------------

NUM_CHAIRS = 3          # Número de sillas en la sala de espera
NUM_CLIENTS = 5         # Cantidad de clientes que llegarán

# -------------------------------------------------------------
# Lock principal
#
# Garantiza exclusión mutua sobre el estado compartido.
# Ningún hilo puede modificar simultáneamente la barbería.
# -------------------------------------------------------------

lock = threading.Lock()

# -------------------------------------------------------------
# Variable de Condición
#
# Permite bloquear y despertar hilos sin realizar espera activa
# (Busy Waiting).
# -------------------------------------------------------------

condition = threading.Condition(lock)

# -------------------------------------------------------------
# Estado compartido de la barbería
# -------------------------------------------------------------

# Número de clientes esperando
waiting = 0

# Cola FIFO de clientes
#
# Se utilizará para:
# - Mantener el orden de llegada.
# - Calcular tiempos de espera.
# - Medir throughput por cliente.
waiting_queue = []

# Estado del barbero
barber_sleeping = False

# Permite finalizar la simulación
shop_open = True

# -------------------------------------------------------------
# Variables de auditoría
# -------------------------------------------------------------

clients_served = 0
clients_rejected = 0

# -------------------------------------------------------------
# Registro de eventos
# -------------------------------------------------------------

start_time = time.perf_counter()

log_lock = threading.Lock()

def log(actor, message):
    """
    Imprime eventos con marca de tiempo.
    Facilita la explicación durante la sustentación.
    """

    with log_lock:

        ts = time.perf_counter() - start_time

        print(f"[T={ts:.4f}s] [{actor}] {message}")

In [190]:
# ===========================================================================
# CHUNK 1,5 — Instrumentación de Métricas
# -------------------------------------------------------------
# 
# Estas estructuras almacenan información durante la ejecución
# para calcular posteriormente:
#
# - Tiempo de espera.
# - Throughput.
# - Tiempo total de simulación.
#
# No afectan la sincronización; únicamente registran eventos
# para el análisis del Frente 5.
# -------------------------------------------------------------
# ===========================================================================

# Momento en que llega cada cliente
arrival_times = {}

# Momento en que inicia su atención
service_times = {}

# Tiempo de espera individual
waiting_times = []

# Tiempo total de la simulación
simulation_start = None
simulation_end = None

## Implementación del Hilo del Barbero

Este bloque implementa el comportamiento del hilo que representa al barbero. El barbero permanece ejecutándose mientras la barbería se encuentre abierta y utiliza una **Variable de Condición** para suspender su ejecución cuando no existen clientes en espera.

Siguiendo la recomendación de **OSTEP (Capítulo 30)**, la espera se realiza dentro de un ciclo `while`, garantizando que la condición sea verificada nuevamente después de cada despertar. De esta manera, la implementación evita despertares espurios y elimina la posibilidad del problema conocido como **Lost Wakeup**.

Finalmente, el corte de cabello se realiza fuera de la sección crítica para permitir que otros clientes puedan llegar a la barbería de forma concurrente sin bloquear el acceso a los recursos compartidos.

In [191]:
# ===========================================================================
# Chunk 2: Hilo del Barbero
# ===========================================================================

def barber_correct():
    """
    Implementación correcta del hilo del barbero.

    Se utiliza una Variable de Condición (Condition) protegida por un Lock.

    La combinación:

            Lock + Condition + while

    evita el problema de Lost Wakeup descrito en OSTEP
    (Capítulo 30).
    """

    global waiting
    global waiting_queue
    global barber_sleeping
    global clients_served
    global shop_open

    global service_times
    global arrival_times
    global waiting_times

    log("BARBERO", "Inicio de turno.")

    while True:

        # ---------------------------------------------------------
        # Sección crítica
        #
        # Solo un hilo puede acceder al estado compartido
        # (waiting, barber_sleeping, etc.) al mismo tiempo.
        # ---------------------------------------------------------

        with condition:

            # -----------------------------------------------------
            # CONDICIÓN DE ESPERA
            #
            # Mientras no existan clientes y la barbería permanezca
            # abierta, el barbero libera el Lock y entra en espera.
            #
            # Se utiliza WHILE (y no IF) por dos razones:
            #
            # 1. Evitar despertares espurios.
            # 2. Revalidar la condición cuando recupera el Lock.
            #
            # (OSTEP - Capítulo 30)
            # -----------------------------------------------------

            while len(waiting_queue) == 0 and shop_open:

                barber_sleeping = True

                log(
                    "BARBERO",
                    "No hay clientes. Esperando..."
                )

                condition.wait()

            # -----------------------------------------------------
            # CONDICIÓN DE PARADA
            #
            # El hilo principal marcará shop_open=False cuando
            # todos los clientes hayan finalizado.
            #
            # Si ya no existen clientes pendientes,
            # el barbero termina ordenadamente.
            # -----------------------------------------------------

            if not shop_open and len(waiting_queue) == 0:

                log(
                    "BARBERO",
                    "Fin del turno."
                )

                break

            # -----------------------------------------------------
            # Existe al menos un cliente esperando.
            #
            # El barbero toma un cliente de la sala.
            # -----------------------------------------------------

            barber_sleeping = False
            #verificacion de invariantes
            verificar_invariantes(
            len(waiting_queue),
            barber_sleeping,
            NUM_CHAIRS
        )

            # ---------------------------------------------------------
            # El primer cliente de la cola pasa a ser atendido.
            #
            # La cola FIFO garantiza el orden de llegada.
            # ---------------------------------------------------------

            current_client = waiting_queue.pop(0) # Es una orden que saca de la fila al elemento que está en la posición número 0 (el primero en llegar).
                # RAFAGAS
            verificar_invariantes(
                len(waiting_queue),
                barber_sleeping,
                NUM_CHAIRS
            )
            # Registrar el inicio de la atención
            service_times[current_client] = time.perf_counter()

            # Calcular tiempo de espera del cliente
            espera = (
                service_times[current_client]
                - arrival_times[current_client]
            )

            waiting_times.append(espera)

            clients_served += 1

            log(
                "BARBERO",
                f"Comienza a atender al Cliente {current_client}. "
                f"Clientes esperando = {len(waiting_queue)}"
            )
        # ---------------------------------------------------------
        # CORTE DE CABELLO
        #
        # El corte ocurre FUERA de la sección crítica.
        #
        # De esta forma otros clientes pueden seguir llegando
        # mientras el barbero trabaja.
        # ---------------------------------------------------------

        time.sleep(random.uniform(0.10,0.30))
        #time.sleep(0.20)

        log(
            "BARBERO",
            "Corte finalizado."
        )

In [192]:
# ===========================================================================
# Chunk 3: Hilo del Cliente
# ===========================================================================

def client_correct(client_id, arrival_delay):
    """
    Implementación correcta del hilo cliente.

    Cada cliente llega a la barbería de manera concurrente,
    intenta ocupar una silla de espera y, si el barbero se
    encuentra dormido, lo despierta mediante notify().

    Todas las operaciones sobre el estado compartido se
    realizan dentro de la sección crítica protegida por
    el Lock asociado a la Variable de Condición.
    """

    global waiting
    global waiting_queue
    global barber_sleeping
    global clients_rejected

    global arrival_times

    # ---------------------------------------------------------
    # Simulación del tiempo de llegada del cliente
    # ---------------------------------------------------------

    time.sleep(arrival_delay)

    log(
        f"CLIENTE-{client_id}",
        "Llegando a la barbería."
    )

    # Registrar el instante de llegada del cliente
    arrival_times[client_id] = time.perf_counter()

    # ---------------------------------------------------------
    # Sección crítica
    # ---------------------------------------------------------

    with condition:

        # -----------------------------------------------------
        # Verificación de espacio disponible.
        #
        # Si la sala está llena, el cliente abandona
        # la barbería inmediatamente.
        # -----------------------------------------------------

        if len(waiting_queue) >= NUM_CHAIRS:

            clients_rejected += 1

            log(
                f"CLIENTE-{client_id}",
                "Sala llena. Me retiro."
            )

            return

        # -----------------------------------------------------
        # El cliente ocupa una silla de espera.
        # -----------------------------------------------------

       
        waiting_queue.append(client_id)
        

        log(
            f"CLIENTE-{client_id}",
            f"Esperando. Clientes en sala = {len(waiting_queue)}"
        )

        # -----------------------------------------------------
        # Si el barbero estaba dormido,
        # el cliente lo despierta.
        #
        # notify() solamente informa que la condición
        # cambió. El barbero volverá a verificar
        # la cola de espera (waiting_queue) dentro
        # del ciclo while antes de continuar.
        # -----------------------------------------------------

        if barber_sleeping:

            log(
                f"CLIENTE-{client_id}",
                "Despertando al barbero."
            )

            condition.notify()

    # ---------------------------------------------------------
    # El cliente termina su ejecución.
    #
    # La atención será realizada por el hilo del barbero.
    # -----------------------------------------------------

    log(
        f"CLIENTE-{client_id}",
        "Esperando su turno."
    )

## Creación de los Hilos

En este bloque se crean los hilos que participarán en la simulación. Se instancia un único hilo para representar al barbero y varios hilos para representar a los clientes, cada uno con un tiempo de llegada aleatorio para simular un escenario de concurrencia.

En esta etapa únicamente se crean los hilos; la ejecución de la simulación se realizará en el siguiente bloque.

In [193]:
# ===========================================================================
# Chunk 4: Creación de los Hilos
# ===========================================================================

print("=== SIMULACIÓN: IMPLEMENTACIÓN CORRECTA DEL PELUQUERO DORMIDO ===\n")

# -------------------------------------------------------------
# Reinicio del estado compartido
# -------------------------------------------------------------

waiting = 0
waiting_queue = []

barber_sleeping = False
shop_open = True

clients_served = 0
clients_rejected = 0

start_time = time.perf_counter()

# -------------------------------------------------------------
# Declaración del hilo del barbero
#
# Existe un único barbero durante toda la simulación.
# Permanecerá ejecutándose hasta que la barbería cierre.
# -------------------------------------------------------------

barber = threading.Thread(
    target=barber_correct,
    name="BARBERO"
)

# -------------------------------------------------------------
# Declaración de los hilos clientes
#
# Cada cliente es un hilo independiente que llega
# después de un pequeño retardo aleatorio.
# -------------------------------------------------------------

clients = []

for i in range(NUM_CLIENTS):

    arrival = random.uniform(0.0,0.5)

    client = threading.Thread(
        target=client_correct,
        args=(i + 1, arrival),
        name=f"CLIENTE-{i+1}"
    )

    clients.append(client)

=== SIMULACIÓN: IMPLEMENTACIÓN CORRECTA DEL PELUQUERO DORMIDO ===



## Ejecución de la Simulación

En este bloque se inicia la ejecución concurrente de todos los hilos. Primero se inicia el hilo del barbero y posteriormente los hilos clientes.

El hilo principal utiliza `join()` para esperar la finalización de todos los clientes. Una vez atendidos o rechazados, se cambia la condición de parada (`shop_open = False`) y se despierta al barbero mediante `notify()` para que finalice su ejecución de forma ordenada.

Finalmente se imprime un resumen con el número de clientes atendidos y rechazados.

In [194]:
# ===========================================================================
# Chunk 5: Ejecución de la Simulación
# ===========================================================================



# -------------------------------------------------------------
# Inicio del hilo del barbero
# -------------------------------------------------------------
 #Inicio de la medición
simulation_start = time.perf_counter()
# Iniciar el hilo del barbero
barber.start()


# -------------------------------------------------------------
# Inicio de los clientes
# -------------------------------------------------------------

for client in clients:
    client.start()

# -------------------------------------------------------------
# Esperar que todos los clientes finalicen
#
# join() garantiza que el hilo principal espere
# hasta que cada cliente termine su ejecución.
# -------------------------------------------------------------

for client in clients:
    client.join()

# -------------------------------------------------------------
# Condición de parada
#
# Una vez finalizados todos los clientes,
# la barbería deja de aceptar nuevas llegadas.
#
# Se despierta al barbero para que salga del wait()
# y finalice su ejecución.
# -------------------------------------------------------------

with condition:

    shop_open = False

    condition.notify()

# -------------------------------------------------------------
# Esperar la finalización del barbero
# -------------------------------------------------------------

barber.join()
# Final de la medición
simulation_end = time.perf_counter()
simulation_time = simulation_end - simulation_start


# -------------------------------------------------------------
# Resumen de la simulación
# -------------------------------------------------------------

print("\n==========================================")
print("RESULTADOS DE LA SIMULACIÓN")
print("==========================================")

print(f"Clientes atendidos : {clients_served}")
print(f"Clientes en cola : {waiting_queue}")
print(f"Clientes rechazados: {clients_rejected}")
print(f"Clientes restantes : {waiting}")

print("\nImplementación finalizada correctamente.")


[T=0.0043s] [BARBERO] Inicio de turno.
[T=0.0045s] [BARBERO] No hay clientes. Esperando...
[T=0.0851s] [CLIENTE-5] Llegando a la barbería.
[T=0.0852s] [CLIENTE-5] Esperando. Clientes en sala = 1
[T=0.0852s] [CLIENTE-5] Despertando al barbero.
[T=0.0852s] [CLIENTE-5] Esperando su turno.
[T=0.0853s] [BARBERO] Comienza a atender al Cliente 5. Clientes esperando = 0
[T=0.1444s] [CLIENTE-3] Llegando a la barbería.
[T=0.1444s] [CLIENTE-3] Esperando. Clientes en sala = 1
[T=0.1444s] [CLIENTE-3] Esperando su turno.
[T=0.1884s] [CLIENTE-1] Llegando a la barbería.
[T=0.1888s] [CLIENTE-1] Esperando. Clientes en sala = 2
[T=0.1888s] [CLIENTE-1] Esperando su turno.
[T=0.2006s] [CLIENTE-2] Llegando a la barbería.
[T=0.2008s] [CLIENTE-2] Esperando. Clientes en sala = 3
[T=0.2008s] [CLIENTE-2] Esperando su turno.
[T=0.2646s] [CLIENTE-4] Llegando a la barbería.
[T=0.2650s] [CLIENTE-4] Sala llena. Me retiro.
[T=0.3582s] [BARBERO] Corte finalizado.
[T=0.3582s] [BARBERO] Comienza a atender al Cliente 3. C

In [195]:
print(simulation_start)
print(simulation_end)

37467.457649416
37468.465968083


 # Frente 4 — Pruebas de estrés y verificación: Harness con aserciones y casos adversos que intenten romper la solución.

# Frente 4 — Pruebas de Estrés y Verificación

##  Objetivo del Harness

En este frente se verifica que la implementación correcta preserve los invariantes definidos en el Frente 2 bajo distintos escenarios de concurrencia.

El propósito del *Harness* es ejecutar automáticamente varios casos de prueba que intentan llevar el sistema a situaciones límite, comprobando mediante aserciones que la solución mantiene las propiedades de **Seguridad (Safety)**, **Vivacidad (Liveness)** y **Equidad (Fairness)**.

Cada escenario ejecutará la simulación con diferentes patrones de llegada de clientes y verificará que:

- Nunca existan más clientes esperando que sillas disponibles.
- El barbero nunca permanezca dormido existiendo clientes en espera.
- Todos los clientes sean correctamente atendidos o rechazados.
- La simulación finalice sin producir interbloqueos (*Deadlocks*).

In [196]:
# ===========================================================================
# CHUNK 2 — FUNCIÓN GENERAL DE SIMULACIÓN
# ===========================================================================

def ejecutar_simulacion(num_clientes, llegada_min, llegada_max):

    global waiting
    global waiting_queue
    global barber_sleeping
    global shop_open
    global clients_served
    global clients_rejected

    # NUEVO
    global simulation_start
    global simulation_end
    global arrival_times
    global service_times
    global waiting_times

    print("\n" + "="*60)
    print(f"Simulación con {num_clientes} clientes")
    print("="*60)

    # Reiniciar estado compartido

    waiting = 0
    barber_sleeping = False
    shop_open = True

    clients_served = 0
    clients_rejected = 0

     # NUEVO: reiniciar estructuras de medición
    arrival_times = {}
    service_times = {}
    waiting_times = []

    # Crear hilo del barbero

    # -------------------------------------------------------------
    # Inicio de la medición
    # -------------------------------------------------------------
    simulation_start = time.perf_counter()

    # Crear hilo del barbero
    barber = threading.Thread(target=barber_correct)

    barber.start()

    # Crear clientes

    clientes = []

    for i in range(num_clientes):

        llegada = random.uniform(
            llegada_min,
            llegada_max
        )

        hilo = threading.Thread(
            target=client_correct,
            args=(i+1, llegada)
        )

        clientes.append(hilo)

        hilo.start()

    # Esperar clientes

    for hilo in clientes:

        hilo.join()

    # Cerrar barbería

    with condition:

        shop_open = False

        condition.notify()

    barber.join()

    # -------------------------------------------------------------
    # Fin de la medición
    # -------------------------------------------------------------
    simulation_end = time.perf_counter()

    simulation_time = simulation_end - simulation_start

    print("\nResumen")

    print(f"Atendidos : {clients_served}")

    print(f"Rechazados: {clients_rejected}")

    print(f"Esperando : {len(waiting_queue)}")

    verificar_invariantes(
        len(waiting_queue),
        barber_sleeping
    )

    # -------------------------------------------------------------
    # Verificación final de los invariantes
    # -------------------------------------------------------------
    verificar_invariantes(
        len(waiting_queue),
        barber_sleeping,
        NUM_CHAIRS
    )
    print(f"Verificaciones ejecutadas : {total_verificaciones}")

    

    # -------------------------------------------------------------
    # Retornar las métricas de la simulación
    # -------------------------------------------------------------
    return {
        "simulation_time": simulation_time,
        "clients_served": clients_served,
        "clients_rejected": clients_rejected,
        "waiting_times": waiting_times
    }
    print("✔ Simulación finalizada correctamente.")


### Caso base

In [197]:
resultado = ejecutar_simulacion(
    num_clientes=10,
    llegada_min=0,
    llegada_max=0.2
)

print(resultado)


Simulación con 10 clientes
[T=1.0363s] [BARBERO] Inicio de turno.
[T=1.0363s] [BARBERO] No hay clientes. Esperando...
[T=1.0370s] [CLIENTE-1] Llegando a la barbería.
[T=1.0370s] [CLIENTE-1] Esperando. Clientes en sala = 1
[T=1.0370s] [CLIENTE-1] Despertando al barbero.
[T=1.0370s] [CLIENTE-1] Esperando su turno.
[T=1.0372s] [BARBERO] Comienza a atender al Cliente 1. Clientes esperando = 0
[T=1.0510s] [CLIENTE-5] Llegando a la barbería.
[T=1.0511s] [CLIENTE-5] Esperando. Clientes en sala = 1
[T=1.0511s] [CLIENTE-5] Esperando su turno.
[T=1.0530s] [CLIENTE-10] Llegando a la barbería.
[T=1.0530s] [CLIENTE-10] Esperando. Clientes en sala = 2
[T=1.0530s] [CLIENTE-10] Esperando su turno.
[T=1.0591s] [CLIENTE-7] Llegando a la barbería.
[T=1.0592s] [CLIENTE-7] Esperando. Clientes en sala = 3
[T=1.0592s] [CLIENTE-7] Esperando su turno.
[T=1.0737s] [CLIENTE-6] Llegando a la barbería.
[T=1.0738s] [CLIENTE-6] Sala llena. Me retiro.
[T=1.0758s] [CLIENTE-2] Llegando a la barbería.
[T=1.0758s] [CLIE

In [198]:
print("CASO 1")

ejecutar_simulacion(

    num_clientes=5,

    llegada_min=0.2,

    llegada_max=0.5

)

CASO 1

Simulación con 5 clientes
[T=1.8642s] [BARBERO] Inicio de turno.
[T=1.8642s] [BARBERO] No hay clientes. Esperando...
[T=2.1550s] [CLIENTE-1] Llegando a la barbería.
[T=2.1566s] [CLIENTE-1] Esperando. Clientes en sala = 1
[T=2.1566s] [CLIENTE-1] Despertando al barbero.
[T=2.1567s] [CLIENTE-1] Esperando su turno.
[T=2.1568s] [BARBERO] Comienza a atender al Cliente 1. Clientes esperando = 0
[T=2.2019s] [CLIENTE-2] Llegando a la barbería.
[T=2.2020s] [CLIENTE-2] Esperando. Clientes en sala = 1
[T=2.2020s] [CLIENTE-2] Esperando su turno.
[T=2.2211s] [CLIENTE-3] Llegando a la barbería.
[T=2.2211s] [CLIENTE-3] Esperando. Clientes en sala = 2
[T=2.2211s] [CLIENTE-3] Esperando su turno.
[T=2.2911s] [CLIENTE-4] Llegando a la barbería.
[T=2.2914s] [CLIENTE-4] Esperando. Clientes en sala = 3
[T=2.2914s] [CLIENTE-4] Esperando su turno.
[T=2.3241s] [BARBERO] Corte finalizado.
[T=2.3243s] [BARBERO] Comienza a atender al Cliente 2. Clientes esperando = 2
[T=2.3515s] [CLIENTE-5] Llegando a la b

{'simulation_time': 1.0557185420038877,
 'clients_served': 5,
 'clients_rejected': 0,
 'waiting_times': [0.0001998329971684143,
  0.12235216599947307,
  0.2137068750016624,
  0.26766137500089826,
  0.42996170799597166]}

### LLegada al mismo Tiempo

In [199]:
print("CASO 2")

ejecutar_simulacion(

    num_clientes=5,

    llegada_min=0,

    llegada_max=0

)

CASO 2

Simulación con 5 clientes
[T=2.9342s] [BARBERO] Inicio de turno.
[T=2.9343s] [BARBERO] No hay clientes. Esperando...
[T=2.9345s] [CLIENTE-1] Llegando a la barbería.
[T=2.9346s] [CLIENTE-1] Esperando. Clientes en sala = 1
[T=2.9346s] [CLIENTE-1] Despertando al barbero.
[T=2.9346s] [CLIENTE-1] Esperando su turno.
[T=2.9347s] [CLIENTE-2] Llegando a la barbería.
[T=2.9347s] [CLIENTE-2] Esperando. Clientes en sala = 2
[T=2.9347s] [CLIENTE-2] Despertando al barbero.
[T=2.9347s] [CLIENTE-2] Esperando su turno.
[T=2.9351s] [CLIENTE-4] Llegando a la barbería.
[T=2.9351s] [CLIENTE-4] Esperando. Clientes en sala = 3
[T=2.9351s] [CLIENTE-4] Despertando al barbero.
[T=2.9351s] [CLIENTE-4] Esperando su turno.
[T=2.9351s] [CLIENTE-3] Llegando a la barbería.
[T=2.9351s] [CLIENTE-3] Sala llena. Me retiro.
[T=2.9352s] [BARBERO] Comienza a atender al Cliente 1. Clientes esperando = 2
[T=2.9355s] [CLIENTE-5] Llegando a la barbería.
[T=2.9355s] [CLIENTE-5] Esperando. Clientes en sala = 3
[T=2.9355s

{'simulation_time': 0.6387170420057373,
 'clients_served': 4,
 'clients_rejected': 1,
 'waiting_times': [0.0006137919990578666,
  0.14346358300099382,
  0.3359679999994114,
  0.4750066249980591]}

### Sala llena

In [200]:
print("CASO 3")

NUM_CHAIRS = 2

ejecutar_simulacion(

    num_clientes=10,

    llegada_min=0,

    llegada_max=0.2

)

CASO 3

Simulación con 10 clientes
[T=3.5852s] [BARBERO] Inicio de turno.
[T=3.5852s] [BARBERO] No hay clientes. Esperando...
[T=3.6902s] [CLIENTE-3] Llegando a la barbería.
[T=3.6917s] [CLIENTE-3] Esperando. Clientes en sala = 1
[T=3.6917s] [CLIENTE-3] Despertando al barbero.
[T=3.6918s] [CLIENTE-3] Esperando su turno.
[T=3.6919s] [BARBERO] Comienza a atender al Cliente 3. Clientes esperando = 0
[T=3.6957s] [CLIENTE-9] Llegando a la barbería.
[T=3.6958s] [CLIENTE-9] Esperando. Clientes en sala = 1
[T=3.6958s] [CLIENTE-9] Esperando su turno.
[T=3.7045s] [CLIENTE-7] Llegando a la barbería.
[T=3.7045s] [CLIENTE-7] Esperando. Clientes en sala = 2
[T=3.7045s] [CLIENTE-7] Esperando su turno.
[T=3.7124s] [CLIENTE-4] Llegando a la barbería.
[T=3.7125s] [CLIENTE-4] Sala llena. Me retiro.
[T=3.7338s] [CLIENTE-2] Llegando a la barbería.
[T=3.7340s] [CLIENTE-2] Sala llena. Me retiro.
[T=3.7341s] [CLIENTE-6] Llegando a la barbería.
[T=3.7342s] [CLIENTE-6] Sala llena. Me retiro.
[T=3.7352s] [CLIENT

{'simulation_time': 0.7235767910024151,
 'clients_served': 3,
 'clients_rejected': 7,
 'waiting_times': [0.00015479099965887144,
  0.291868959000567,
  0.42899645799479913]}

### Estrés

In [201]:
print("CASO 4")

NUM_CHAIRS = 5

resultado = ejecutar_simulacion(
    num_clientes=100,
    llegada_min=0,
    llegada_max=0.02
)

CASO 4

Simulación con 100 clientes
[T=4.3224s] [BARBERO] Inicio de turno.
[T=4.3224s] [BARBERO] No hay clientes. Esperando...
[T=4.3257s] [CLIENTE-27] Llegando a la barbería.
[T=4.3257s] [CLIENTE-27] Esperando. Clientes en sala = 1
[T=4.3258s] [CLIENTE-27] Despertando al barbero.
[T=4.3258s] [CLIENTE-27] Esperando su turno.
[T=4.3259s] [BARBERO] Comienza a atender al Cliente 27. Clientes esperando = 0
[T=4.3262s] [CLIENTE-18] Llegando a la barbería.
[T=4.3262s] [CLIENTE-18] Esperando. Clientes en sala = 1
[T=4.3262s] [CLIENTE-18] Esperando su turno.
[T=4.3267s] [CLIENTE-28] Llegando a la barbería.
[T=4.3267s] [CLIENTE-28] Esperando. Clientes en sala = 2
[T=4.3268s] [CLIENTE-28] Esperando su turno.
[T=4.3272s] [CLIENTE-29] Llegando a la barbería.
[T=4.3272s] [CLIENTE-29] Esperando. Clientes en sala = 3
[T=4.3272s] [CLIENTE-29] Esperando su turno.
[T=4.3273s] [CLIENTE-11] Llegando a la barbería.
[T=4.3273s] [CLIENTE-11] Esperando. Clientes en sala = 4
[T=4.3273s] [CLIENTE-11] Esperando 

# Frente 5 — Medición y Análisis del Desempeño


## Objetivo de la Medición

Hasta este punto se ha demostrado que la solución elimina el problema del **Lost Wakeup** y preserva los invariantes del sistema bajo distintos escenarios de concurrencia.

En este frente se realizan mediciones cuantitativas para evaluar el desempeño de la implementación. Se analizarán tres métricas principales:

- **Tiempo de espera de los clientes:** tiempo transcurrido desde que un cliente llega a la barbería hasta que comienza a ser atendido.
- **Throughput:** cantidad de clientes atendidos por segundo durante la simulación.
- **Contención:** número de clientes que debieron esperar debido a que el recurso compartido (el barbero) ya estaba ocupado.

Estas métricas permiten respaldar experimentalmente que la solución no solo es correcta desde el punto de vista de la sincronización, sino también eficiente bajo diferentes niveles de carga.

### Estructuras donde almacenaremos la información.

In [202]:
# ===========================================================================
# CHUNK 3 — Cálculo de Métricas
# ===========================================================================

import statistics

# Tiempo total de la simulación
simulation_time = simulation_end - simulation_start

# Throughput
throughput = clients_served / simulation_time

# Tiempo promedio de espera
average_wait = statistics.mean(waiting_times)

# Tiempo máximo de espera
max_wait = max(waiting_times)

# Tiempo mínimo de espera
min_wait = min(waiting_times)

print("\n========================================")
print("MÉTRICAS DE DESEMPEÑO")
print("========================================")

print(f"Tiempo total de simulación : {simulation_time:.3f} s")
print(f"Clientes atendidos         : {clients_served}")
print(f"Clientes rechazados        : {clients_rejected}")
print(f"Throughput                 : {throughput:.2f} clientes/s")
print(f"Tiempo promedio espera     : {average_wait:.3f} s")
print(f"Tiempo máximo espera       : {max_wait:.3f} s")
print(f"Tiempo mínimo espera       : {min_wait:.3f} s")


MÉTRICAS DE DESEMPEÑO
Tiempo total de simulación : 1.098 s
Clientes atendidos         : 6
Clientes rechazados        : 94
Throughput                 : 5.46 clientes/s
Tiempo promedio espera     : 0.494 s
Tiempo máximo espera       : 0.920 s
Tiempo mínimo espera       : 0.000 s


## Análisis de Resultados

Las métricas obtenidas permiten analizar el comportamiento de la implementación bajo carga concurrente.

- Un **Throughput** elevado indica que el barbero mantiene un ritmo constante de atención sin bloqueos.
- Un **Tiempo promedio de espera** bajo refleja una utilización eficiente del recurso compartido.
- La diferencia entre el tiempo mínimo y máximo de espera muestra el efecto de la concurrencia y de la ocupación de la sala de espera.
- La ausencia de tiempos anómalos confirma que la implementación evita interbloqueos (*Deadlocks*) y elimina el problema del **Lost Wakeup** observado en el Frente 1.

En conjunto, estas mediciones respaldan experimentalmente que la solución basada en **Lock + Condition + while** cumple tanto los requisitos funcionales como las propiedades de sincronización exigidas por el problema.

In [203]:
# ===========================================================================
# CHUNK 5 — TABLA RESUMEN DE MÉTRICAS
# ===========================================================================

from IPython.display import display
import pandas as pd

# Construcción de la tabla de resultados
tabla_resultados = pd.DataFrame({

    "Métrica": [

        "Clientes atendidos",
        "Clientes rechazados",
        "Tiempo total de simulación (s)",
        "Throughput (clientes/s)",
        "Tiempo promedio de espera (s)",
        "Tiempo máximo de espera (s)",
        "Tiempo mínimo de espera (s)"

    ],

    "Valor": [

        clients_served,
        clients_rejected,
        round(simulation_time,3),
        round(throughput,2),
        round(average_wait,3),
        round(max_wait,3),
        round(min_wait,3)

    ]

})

print("\n==============================")
print("RESUMEN DE RESULTADOS")
print("==============================\n")

display(tabla_resultados)


RESUMEN DE RESULTADOS



,Métrica,Valor
0,Clientes atendidos,6.000
1,Clientes rechazados,94.000
2,Tiempo total de simulación (s),1.098
3,Throughput (clientes/s),5.460
4,Tiempo promedio de espera (s),0.494
5,Tiempo máximo de espera (s),0.920
6,Tiempo mínimo de espera (s),0.000


# ADR 1 — Uso de Monitor (Lock + Condition)


El problema requiere coordinar múltiples hilos que comparten una sala de espera y un único barbero.

Problema

Era necesario evitar condiciones de carrera sobre el estado compartido de la barbería.

Alternativas
Variables compartidas sin sincronización.
Lock únicamente.
Semáforos.
Monitor (Lock + Condition).
Decisión

Se implementó un Monitor utilizando threading.Lock y threading.Condition.

Justificación

El Monitor garantiza exclusión mutua y permite dormir y despertar hilos sin espera activa (Busy Waiting).

Evidencia

La solución elimina el problema de Lost Wakeup demostrado en el Frente 1.

 * Frente 3 → Chunk 1

 lock = threading.Lock()

 condition = threading.Condition(lock)

# ADR 2 — Uso de while en Condition.wait()


Los hilos permanecen bloqueados hasta que cambia el estado de la barbería.

Problema

Si se utiliza if, un despertar espurio puede hacer que un hilo continúe cuando la condición aún no se cumple.

Decisión

Toda espera sobre una Variable de Condición se implementó mediante:

* Frente 3 → Chunk 2

while len(waiting_queue) == 0 and shop_open:

    barber_sleeping = True

    condition.wait()

# ADR 3 — Cola FIFO


Era necesario mantener el orden de llegada de los clientes.

Problema

Un simple contador (waiting) no permitía identificar qué cliente debía ser atendido.

Decisión

Se implementó una cola FIFO (waiting_queue).

Justificación

Permite:

mantener equidad;
calcular tiempos de espera;
medir throughput por cliente.
Evidencia

Los clientes fueron atendidos respetando el orden de llegada.

Código asociado

* Frente 3 → Chunk 1

waiting_queue = []

* Chunk 2

current_client = waiting_queue.pop(0)

* Chunk 3

waiting_queue.append(client_id)
Justificación



# ADR 4 — Instrumentación

No era posible calcular tiempos de espera sin registrar eventos.

Decisión

Se instrumentó el código registrando:

llegada;
inicio del servicio;
duración total.
Evidencia

Fue posible calcular:

Throughput.
Tiempo promedio.
Tiempo máximo.

* Frente 3 → Chunk 1.5

arrival_times = {}

service_times = {}

waiting_times = []

# ADR 5 — Invariantes Ejecutables

Los invariantes no podían verificarse únicamente al finalizar la simulación.

Decisión

Los invariantes se implementaron mediante assert.

Evidencia

Cada modificación del estado ejecuta las verificaciones automáticamente.

# ADR 6 — Harness de pruebas


El sistema debía probarse bajo diferentes cargas.


Se creó una función general:

ejecutar_simulacion()

que permite variar:

cantidad de clientes;
velocidad de llegada;
estrés.
Evidencia

Fue posible ejecutar escenarios normales y adversos reutilizando la misma infraestructura.

* Frente 4

def ejecutar_simulacion(...)

| Monitor                                 | Semáforos                                               |
| --------------------------------------- | ------------------------------------------------------- |
| Expresa condiciones del estado          | Expresa disponibilidad de recursos                      |
| Más legible                             | Más bajo nivel                                          |
| `wait()` libera el Lock automáticamente | Hay que controlar `acquire()` y `release()` manualmente |
| Menor probabilidad de errores           | Mayor riesgo de errores de sincronización               |
| Recomendado por OSTEP para este caso    | Solución clásica de Downey                              |
